In [1]:
from __future__ import annotations

import os
import re
import json
import math
import random
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor, get_cosine_schedule_with_warmup
from accelerate import Accelerator

In [ ]:
# os.environ['HF_TOKEN']= ''

In [3]:
# import glob

# path_dataset = '/workspace/Med_VQA/datasets/path-vqa'
# img_paths = glob.glob(
#     os.path.join(path_dataset, "**", "*.png"),
#     recursive=True
# )+glob.glob(
#     os.path.join(path_dataset, "**", "*.jpg"),
#     recursive=True
# )

In [4]:
# data = json.load(open(glob.glob(
#     os.path.join(path_dataset, "**", "*train*.json"),
#     recursive=True
# )[0],'r'))

In [5]:
# def map_sample2img(samples,imge_fullpaths):
#     img_name = 'img_name' if samples[0].get('img_name','image_name') != 'image_name' else 'image_name'
#     cnt = samples[0][img_name].strip().strip('.').strip('/').count('/')
    
#     return {
#         '/'.join(img_path.split('/')[-(cnt+1):]): img_path for img_path in imge_fullpaths
#     }
    

In [6]:
# from PIL import Image
# class PreDataset(Dataset):
#     def __init__(self, path_dataset,type_split='train'):
#         img_paths = glob.glob(
#                             os.path.join(path_dataset, "**", "*.png"),
#                             recursive=True
#                         )+glob.glob(
#                             os.path.join(path_dataset, "**", "*.jpg"),
#                             recursive=True
#                         )
#         self.samples = list(filter(lambda x : x.get("q_lang","en")=="en",
#             json.load(open(glob.glob(
#                                     os.path.join(path_dataset, "**", f"{type_split}*.json"),
#                                     recursive=True
#                                 )[0],'r'))))
#         self.map_imgName = map_sample2img(self.samples,img_paths)
#     def __len__(self):
#         return len(self.samples)
#     def __getitem__(self,index):
#         sample_i = self.samples[index]
#         img_name = 'img_name' if sample_i.get('img_name','image_name') != 'image_name' else 'image_name'
#         img_path = self.map_imgName[sample_i[img_name].strip().strip('.').strip('/')]
#         question = sample_i['question']
#         answer = sample_i['answer']
#         answer_type = sample_i['answer_type']
#         return {
#             'image':Image.open(img_path),
#             'question': question,
#             'answer':answer,
#             'answer_type':answer_type
#         }


In [7]:
import sys
sys.path.append('/workspace/Med_VQA/src')

In [8]:
from data.dataloaders import *

In [9]:
from __future__ import annotations

from dataclasses import dataclass, fields
from pathlib import Path
from typing import Optional, Any, Dict

import yaml


@dataclass
class TrainConfig:
    dataset_name: str = "flaviagiammarino/vqa-rad"
    model_name: str = "flaviagiammarino/pubmed-clip-vit-base-patch32"
    model_type: str = "pubmedclip_ban"

    output_dir: str = "runs/pubmedclip_ban"
    seed: int = 42

    epochs: int = 40
    batch_size: int = 16
    eval_batch_size: int = 64
    num_workers: int = 2
    max_length: int = 77

    num_hid: int = 512
    glimpse: int = 4
    freeze_clip: bool = True

    lr_head: float = 1e-4
    lr_clip: float = 1e-6
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.05
    type_loss_weight: float = 0.2
    class_weight_power: float = 0.5
    mixed_precision: str = "fp16"
    grad_clip: float = 1.0

    min_answer_freq: int = 1
    max_answers: Optional[int] = None
    answer_vocab_source: str = "train_eval"
    filter_eval_unseen_train_answers: bool = False

    filter_invalid_count_answers: bool = False
    count_answer_max_num: Optional[int] = None

    eval_with_type_mask: bool = True


def load_config(path: str | Path, overrides: Optional[Dict[str, Any]] = None) -> TrainConfig:
    path = Path(path)
    data = yaml.safe_load(path.read_text(encoding="utf-8")) or {}

    if overrides:
        for key, value in overrides.items():
            if value is not None:
                data[key] = value

    valid_fields = {f.name for f in fields(TrainConfig)}
    unknown = set(data) - valid_fields
    if unknown:
        raise ValueError(f"Unknown config keys: {sorted(unknown)}")

    return TrainConfig(**data)


In [10]:
cfg = TrainConfig(dataset_name= '/workspace/Med_VQA/datasets/path-vqa')

In [11]:
processor = CLIPProcessor.from_pretrained(cfg.model_name)

In [12]:
# processor(
#     ima
# )

In [13]:
next(iter(make_loaders(cfg,processor)['train_loader']))['pixel_values'].shape

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'input_ids': tensor([[49406,   533,   518,  ..., 49407, 49407, 49407],
        [49406,  1897,  1884,  ..., 49407, 49407, 49407],
        [49406,  1897,   651,  ..., 49407, 49407, 49407],
        ...,
        [49406,   533,  1162,  ..., 49407, 49407, 49407],
        [49406,   829,  1897,  ..., 49407, 49407, 49407],
        [49406,   631, 16443,  ..., 49407, 49407, 49407]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'pixel_values': tensor([[[[ 1.9011e+00,  1.9011e+00,  1.9303e+00,  ...,  1.9157e+00,
            1.9157e+00,  1.9157e+00],
          [ 1.9011e+00,  1.9011e+00,  1.9303e+00,  ...,  1.8281e+00,
            1.8281e+00,  1.8281e+00],
          [ 1.9011e+00,  1.9011e+00,  1.9303e+00,  ...,  1.8427e+00,
            1.8281e+00,  1.8281e+00],
          ...,
          [ 1.9157e+00,  1.9157e+0

torch.Size([16, 3, 224, 224])

In [14]:
next(iter(make_loaders(cfg,processor)['train_loader']))

{'input_ids': tensor([[49406,   768,  1897,  ..., 49407, 49407, 49407],
        [49406,   533,   518,  ..., 49407, 49407, 49407],
        [49406,   768,  1897,  ..., 49407, 49407, 49407],
        ...,
        [49406,   768,   631,  ..., 49407, 49407, 49407],
        [49406,   768,   851,  ..., 49407, 49407, 49407],
        [49406,   768,   533,  ..., 49407, 49407, 49407]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'pixel_values': tensor([[[[-1.7339, -1.7047, -1.7485,  ..., -1.7777, -1.7631, -1.7485],
          [-1.7631, -1.7339, -1.7777,  ..., -1.7631, -1.7485, -1.7631],
          [-1.7485, -1.7339, -1.7485,  ..., -1.7631, -1.7777, -1.7777],
          ...,
          [-1.7777, -1.7777, -1.7193,  ..., -1.7777, -1.7777, -1.7631],
          [-1.6171, -1.5587, -1.7193,  ..., -1.7631, -1.6609, -1.66

{'input_ids': tensor([[49406,   768,  1897,  ..., 49407, 49407, 49407],
        [49406,   533,   518,  ..., 49407, 49407, 49407],
        [49406,   768,  1897,  ..., 49407, 49407, 49407],
        ...,
        [49406,   768,   631,  ..., 49407, 49407, 49407],
        [49406,   768,   851,  ..., 49407, 49407, 49407],
        [49406,   768,   533,  ..., 49407, 49407, 49407]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'pixel_values': tensor([[[[-1.7339, -1.7047, -1.7485,  ..., -1.7777, -1.7631, -1.7485],
          [-1.7631, -1.7339, -1.7777,  ..., -1.7631, -1.7485, -1.7631],
          [-1.7485, -1.7339, -1.7485,  ..., -1.7631, -1.7777, -1.7777],
          ...,
          [-1.7777, -1.7777, -1.7193,  ..., -1.7777, -1.7777, -1.7631],
          [-1.6171, -1.5587, -1.7193,  ..., -1.7631, -1.6609, -1.66